In [1]:
import os
import sys
import torch
from pathlib import Path
from itertools import chain
from functools import partial

if "__file__" in globals():
    project_root = Path(__file__).resolve().parent.parent
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if Path.cwd() != project_root:
    os.chdir(project_root)

from torch.utils.data import random_split, DataLoader
from core.config import load_config, print_config
from core.data.loader import setup_dataset, load_finetuning_dataset
from core.data.dataset import finetuning_collate_fn, FinetuningDataset
from core.data.transforms import BeatmapTransform, BeatmapNormalizer
from core.model.bert import BertForContrastiveFineTuning
from core.training.sampler import create_contrastive_sampler
from core.training.finetuner import setup_finetuning
from core.training.checkpoint import CheckpointManager
from core.logger import print_data_summary

In [2]:
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Working directory: {os.getcwd()}")

config = load_config("config", config_dir=".")
print_config(config, "Loaded Fine-Tuning Configuration")

PyTorch version: 2.8.0+cu126
Using device: cuda
Working directory: /home/jessiez/osu_corpora

--- Loaded Fine-Tuning Configuration ---
data:
  max_seq_len: 1023
  val_split: 0.1
  max_samples_per_class:
    aim: 2500
    tech: 2500
model:
  d_model: 512
  n_heads: 8
  n_layers: 6
  dim_feedforward_mult: 4
  dropout: 0.1
  local_attention_window: 256
  cnn_kernel_size: 15
components:
  use_flash_attention: true
  compile_model: true
  compile_mode: default
pretraining:
  db_path: ./data/beatmap_dataset_test/
  batch_size: 8
  num_epochs: 8
  learning_rate: 0.0002
  min_lr: 1.0e-06
  cooldown_type: cosine
  weight_decay: 0.05
  warmup_ratio: 0.1
  stable_ratio: 0.1
  use_amp: true
  checkpoint_dir: ./checkpoints
  grad_clip_norm: 1.0
  gradient_accumulation_steps: 8
  difficulty_loss_weight: 1.0
  mlm_loss_weight: 1.0
  masking_ratio: 0.3
  mean_span_length: 4
  sampling:
    method: kde
    kde_bandwidth: 0.2
    num_bins: 200
finetuning:
  db_path: ./data/beatmap_dataset/
  checkpoint_

In [3]:
colab_url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
DATASET_PATH = setup_dataset(config['finetuning']['db_path'], colab_url)

print(f"Using database: {DATASET_PATH}")

all_beatmaps_data, difficulty_attributes, all_labels, all_tags = load_finetuning_dataset(
    DATASET_PATH,
    max_seq_len=config['data']['max_seq_len'],
    max_samples_per_class=config['data'].get('max_samples_per_class')
)

print_data_summary(all_beatmaps_data)
print(f"Loaded {len(all_labels)} label entries and {len(all_tags)} tag entries.")

Using database: ./data/beatmap_dataset/
Loading fine-tuning dataset with labels and tags...
Loading labels from ./data/labels.json...
Applying max samples per class limit...
Downsampling class 'aim' from 3220 to 2500 samples.
Downsampling class 'tech' from 3548 to 2500 samples.
Found 10363 unique beatmaps for fine-tuning. Loading only this subset...
Loading raw data from Parquet dataset...
Loading beatmap metadata...
Pre-filtered to load 10363 specific beatmap IDs.
Found metadata for 9911 beatmaps. Processing in chunks of 2000...


Processing Chunks: 100%|██████████| 5/5 [00:18<00:00,  3.78s/it]


Consolidating processed chunks...
Loaded raw feature vectors for 9884 beatmaps.
Calculating difficulty attributes (will use cache if available)...


Calculating Attributes: 100%|██████████| 10/10 [00:00<00:00, 87018.76it/s]


Running final data integrity check...


Validating Tensors:  17%|█▋        | 1684/9874 [00:00<00:00, 16837.25it/s]

Validating Tensors:  66%|██████▌   | 6476/9874 [00:00<00:00, 15271.61it/s]

Validating Tensors: 100%|██████████| 9874/9874 [00:00<00:00, 15963.16it/s]


Finished loading and processing all data.
Assembling final labels and tags...
Final fine-tuning dataset size: 9864 beatmaps.
Finished loading fine-tuning dataset.

--- Data Summary ---
Total beatmaps: 9864
Vector dimension: 17
Sequence length - Min: 34, Max: 1023, Avg: 816.7
--------------------
Loaded 9864 label entries and 9864 tag entries.


In [4]:
all_unique_labels = sorted(list(set(chain.from_iterable(all_labels))))
collection_label_encoder = {label: i for i, label in enumerate(all_unique_labels)}

all_unique_tags = sorted(list(set(chain.from_iterable(all_tags))))
user_tag_encoder = {tag: i for i, tag in enumerate(all_unique_tags)}

config['finetuning']['collection_label_classes'] = len(all_unique_labels)
config['finetuning']['user_tag_classes'] = len(all_unique_tags) if user_tag_encoder else 0

sampler_cfg = config['finetuning'].setdefault('sampler', {})
sampler_cfg['use_kde_anchor_sampling'] = False

print(f"Found {len(collection_label_encoder)} unique collection labels.")
print(f"Found {len(user_tag_encoder)} unique user tags.")

val_size = int(len(all_beatmaps_data) * config['data']['val_split'])
train_size = len(all_beatmaps_data) - val_size
indices = list(range(len(all_beatmaps_data)))
train_indices, val_indices = random_split(indices, [train_size, val_size])

print(f"Data split: {len(train_indices)} training, {len(val_indices)} validation")

train_subset_data = [all_beatmaps_data[i] for i in train_indices]
val_subset_data = [all_beatmaps_data[i] for i in val_indices]

train_ratings = []
for i in train_indices:
    ratings_tuple = (
        difficulty_attributes['stars'][i],
        difficulty_attributes['aim'][i],
        difficulty_attributes['speed'][i],
        difficulty_attributes['slider_factor'][i]
    )
    train_ratings.append(ratings_tuple)

val_ratings = []
for i in val_indices:
    ratings_tuple = (
        difficulty_attributes['stars'][i],
        difficulty_attributes['aim'][i],
        difficulty_attributes['speed'][i],
        difficulty_attributes['slider_factor'][i]
    )
    val_ratings.append(ratings_tuple)

train_labels = [all_labels[i] for i in train_indices]
val_labels = [all_labels[i] for i in val_indices]
train_tags = [all_tags[i] for i in train_indices]
val_tags = [all_tags[i] for i in val_indices]

pretrain_checkpoint_manager = CheckpointManager(
    config['pretraining']['checkpoint_dir'],
    model_name=config['model'].get('type', 'model')
)

print("Attempting to load normalization stats from pre-trained checkpoint...")
stats = pretrain_checkpoint_manager.load_normalization_stats()

if stats is None:
    raise FileNotFoundError(
        "Could not load normalization stats from pre-trained checkpoint. "
    )

vector_stats = stats.get('vector_stats', stats)
difficulty_stats = stats.get('difficulty_stats')

normalizer = BeatmapNormalizer(
    vector_stats=vector_stats,
    attribute_stats=difficulty_stats
)
normalizer.update_difficulty_stats(torch.tensor(train_ratings, dtype=torch.float32))
mean, std = normalizer.get_difficulty_stats()
print("Successfully created normalizer from pre-trained vector stats and fine-tuning difficulty ratings.")
print(f"Difficulty stats (mean={mean.tolist()}, std={std.tolist()})")

Found 5 unique collection labels.
Found 0 unique user tags.
Data split: 8878 training, 986 validation
Attempting to load normalization stats from pre-trained checkpoint...
Successfully created normalizer from pre-trained vector stats and fine-tuning difficulty ratings.
Difficulty stats (mean=3.470078229904175, std=2.1766600608825684)


In [5]:
from core.data.transforms import BeatmapAugmenter

train_transform = BeatmapTransform(normalizer, augmenter=BeatmapAugmenter(), augment=True)
val_transform = BeatmapTransform(normalizer, augmenter=None, augment=False)

train_dataset = FinetuningDataset(train_subset_data, train_ratings, train_labels, train_tags, train_transform)
val_dataset = FinetuningDataset(val_subset_data, val_ratings, val_labels, val_tags, val_transform)

if len(train_subset_data) == 0:
    raise ValueError("Training subset is empty after split; cannot configure dataloaders.")

sample_vectors = train_subset_data[0]
if isinstance(sample_vectors, tuple):
    sample_vectors = sample_vectors[0]
if not torch.is_tensor(sample_vectors):
    sample_vectors = torch.as_tensor(sample_vectors)

if sample_vectors.ndim == 0:
    raise ValueError("Sample beatmap vectors must have at least one dimension.")
sample_shape = tuple(sample_vectors.shape)
actual_vector_dim = sample_vectors.shape[-1]

print(f"Detected vector dimension {actual_vector_dim} from sample shape {sample_shape}")
collate_with_args = partial(
    finetuning_collate_fn,
    max_seq_len=config['data']['max_seq_len'],
    vector_dim=actual_vector_dim,
    device=device,
    positive_difficulty_threshold=config['finetuning']['positive_difficulty_threshold']
 )
val_collate_with_args = partial(
    finetuning_collate_fn,
    max_seq_len=config['data']['max_seq_len'],
    vector_dim=actual_vector_dim,
    device=device,
    positive_difficulty_threshold=config['finetuning']['positive_difficulty_threshold']
 )


contrastive_sampler = create_contrastive_sampler(
    labels=train_labels,
    difficulty_ratings=train_ratings,
    config=config
)

train_dataloader = DataLoader(
    train_dataset,
    batch_sampler=contrastive_sampler,
    collate_fn=collate_with_args
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=config['pretraining']['batch_size'],
    shuffle=False,
    collate_fn=val_collate_with_args
)

print(f"Created dataloaders with batch size: {config['pretraining']['batch_size']}")
sample_batch = next(iter(train_dataloader))
vectors, mask, ratings, batch_labels, batch_tags = sample_batch
print(
    f"Sample batch summary - vectors: {vectors.shape}, mask: {mask.shape}, ",
    f"ratings: {ratings.shape}, labels lens: {[len(lbls) for lbls in batch_labels]}, ",
    f"tags lens: {[len(tags) for tags in batch_tags]}",
)


Detected vector dimension 17 from sample shape (1023, 17)
Initializing ContrastiveBatchSampler...
Found 5 usable labels for creating pairs.
Total potential anchors: 8878
Using uniform anchor sampling for anchors (KDE disabled).
Created dataloaders with batch size: 8
Sample batch summary - vectors: torch.Size([8, 1023, 17]), mask: torch.Size([8, 1023]),  ratings: torch.Size([8, 4]), labels lens: [1, 1, 2, 1, 1, 1, 1, 1],  tags lens: [0, 0, 0, 0, 0, 0, 0, 0]


In [6]:
model = BertForContrastiveFineTuning.from_config(config, device)

pretrain_checkpoint_manager = CheckpointManager(
    config['pretraining']['checkpoint_dir'],
    model_name=config['model'].get('type', 'model')
)

if pretrain_checkpoint_manager.checkpoint_exists():
    print("Found pre-trained checkpoint. Loading BERT backbone weights...")
    pretrain_checkpoint = torch.load(
        pretrain_checkpoint_manager.get_checkpoint_path(),
        map_location=device,
        weights_only=False
    )
    
    pretrain_state_dict = pretrain_checkpoint['model_state_dict']
    
    compiled_prefix = '_orig_mod.'
    is_compiled = any(k.startswith(compiled_prefix) for k in pretrain_state_dict.keys())
    
    if is_compiled:
        pretrain_state_dict = {k[len(compiled_prefix):]: v for k, v in pretrain_state_dict.items()}

    bert_state_dict = {k.replace('bert.', ''): v for k, v in pretrain_state_dict.items() if k.startswith('bert.')}
    
    missing, unexpected = model.bert.load_state_dict(bert_state_dict, strict=False)
    print(f"Loaded BERT backbone. Missing keys: {len(missing)}, Unexpected keys: {len(unexpected)}")
else:
    print("WARNING: No pre-trained checkpoint found. Fine-tuning from scratch.")

summary = model.get_summary()
print(f"Model Summary: {summary['total_parameters'] / 1e6:.2f}M parameters")

Compiling Contrastive BERT model with torch.compile...
Found pre-trained checkpoint. Loading BERT backbone weights...
Loaded BERT backbone. Missing keys: 0, Unexpected keys: 0
Model Summary: 33.34M parameters


In [7]:
trainer, checkpoint_manager = setup_finetuning(
    model, train_dataloader, val_dataloader, config, device, normalizer,
    user_tag_encoder, collection_label_encoder
)

start_epoch = 0
if checkpoint_manager.checkpoint_exists():
    try:
        start_epoch, metrics, _, _ = checkpoint_manager.load_checkpoint(
            model, trainer.optimizer, trainer.scheduler, trainer.scaler, device=device
        )
        start_epoch += 1
        print(f"Loaded fine-tuning checkpoint, resuming from epoch {start_epoch + 1}")
        print(f"Previous validation metrics: {metrics}")
    except Exception as e:
        print(f"Could not load fine-tuning checkpoint: {e}. Starting from scratch.")

print(f"Fine-tuning setup complete. Starting from epoch {start_epoch + 1}")

Scheduler: WSD with 166 warmup, 0 stable, 3164 decay steps.
Cooldown type: linear, Min LR Ratio: 0.0333
FineTuningTrainer initialized - AMP: True, Device: cuda, Grad Accum: 4
Effective batch size: 32
Fine-tuning setup complete. Starting from epoch 1


In [8]:
print("--- Starting BERT Fine-Tuning ---")
print(f"Model: {config['model']['n_layers']} layers, {config['model']['d_model']} dimensions")
print(f"Training on {len(train_dataset)} samples, validating on {len(val_dataset)} samples.")
print(f"Total epochs: {config['finetuning']['num_epochs']}")

metrics_tracker = trainer.train(start_epoch)

print("\n--- BERT Fine-Tuning Completed! ---")

--- Starting BERT Fine-Tuning ---
Model: 6 layers, 512 dimensions
Training on 8878 samples, validating on 986 samples.
Total epochs: 3
Starting fine-tuning from epoch 1/3...


Epoch 1 [Train]:   0%|          | 0/1110 [00:00<?, ?it/s]

W1019 04:04:25.478000 601942 .venv/lib/python3.12/site-packages/torch/_inductor/utils.py:1436] [4/0_1] Not enough SMs to use max_autotune_gemm mode


Running validation...


Validation:   0%|          | 0/124 [00:00<?, ?it/s]

...

Calculating embedding metrics on 986 validation samples...
Epoch 2/3 | Time: 515.62s | Train Loss: 0.2207 | Val Loss: 0.2171 | R@1: 0.338 | R@5: 0.716 | R@10: 0.834 | Rho: 0.191 | nDCG@10: 0.497


Epoch 3 [Train]:   0%|          | 0/1110 [00:00<?, ?it/s]

Running validation...


Validation:   0%|          | 0/124 [00:00<?, ?it/s]

Calculating embedding metrics on 986 validation samples...
Epoch 3/3 | Time: 508.92s | Train Loss: 0.1790 | Val Loss: 0.2017 | R@1: 0.278 | R@5: 0.690 | R@10: 0.817 | Rho: 0.155 | nDCG@10: 0.489
Fine-tuning finished.

--- BERT Fine-Tuning Completed! ---
